In [0]:
customers = [
    (1, "Arjun", "Kochi"),
    (2, "Meera", "Chennai"),
    (3, "Rahul", "Bangalore"),
    (4, "Sneha", "Hyderabad"),
    (5, "Vikram", "Delhi")
]
 
df_customers = spark.createDataFrame(customers, ["customer_id","name","city"])
df_customers.show()

+-----------+------+---------+
|customer_id|  name|     city|
+-----------+------+---------+
|          1| Arjun|    Kochi|
|          2| Meera|  Chennai|
|          3| Rahul|Bangalore|
|          4| Sneha|Hyderabad|
|          5|Vikram|    Delhi|
+-----------+------+---------+



In [0]:
orders = [
    (101, 1, "Laptop", 2, 50000, "2024-01-01"),
    (102, 1, "Mobile", 1, 20000, "2024-01-03"),
    (103, 2, "Tablet", 3, 30000, "2024-01-02"),
    (104, 2, "Laptop", 1, 45000, "2024-01-05"),
    (105, 3, "Mobile", 2, 15000, "2024-01-01"),
    (106, 3, "Laptop", 1, 60000, "2024-01-04"),
    (107, 4, "Tablet", 2, 25000, "2024-01-02"),
    (108, 4, "Laptop", 1, 55000, "2024-01-06"),
    (109, 5, "TV", 1, 80000, "2024-01-03"),
    (110, 5, "Mobile", 2, 40000, "2024-01-07")
]
 
df_orders = spark.createDataFrame(
    orders,
    ["order_id","customer_id","product","quantity","price","order_date"]
)
df_orders.show()

+--------+-----------+-------+--------+-----+----------+
|order_id|customer_id|product|quantity|price|order_date|
+--------+-----------+-------+--------+-----+----------+
|     101|          1| Laptop|       2|50000|2024-01-01|
|     102|          1| Mobile|       1|20000|2024-01-03|
|     103|          2| Tablet|       3|30000|2024-01-02|
|     104|          2| Laptop|       1|45000|2024-01-05|
|     105|          3| Mobile|       2|15000|2024-01-01|
|     106|          3| Laptop|       1|60000|2024-01-04|
|     107|          4| Tablet|       2|25000|2024-01-02|
|     108|          4| Laptop|       1|55000|2024-01-06|
|     109|          5|     TV|       1|80000|2024-01-03|
|     110|          5| Mobile|       2|40000|2024-01-07|
+--------+-----------+-------+--------+-----+----------+



# Joins (Day 6)
- Task 1: Join customers + orders


In [0]:
df_customers.join(df_orders, "customer_id","inner").show()

+-----------+------+---------+--------+-------+--------+-----+----------+
|customer_id|  name|     city|order_id|product|quantity|price|order_date|
+-----------+------+---------+--------+-------+--------+-----+----------+
|          1| Arjun|    Kochi|     101| Laptop|       2|50000|2024-01-01|
|          1| Arjun|    Kochi|     102| Mobile|       1|20000|2024-01-03|
|          2| Meera|  Chennai|     103| Tablet|       3|30000|2024-01-02|
|          2| Meera|  Chennai|     104| Laptop|       1|45000|2024-01-05|
|          3| Rahul|Bangalore|     105| Mobile|       2|15000|2024-01-01|
|          3| Rahul|Bangalore|     106| Laptop|       1|60000|2024-01-04|
|          4| Sneha|Hyderabad|     107| Tablet|       2|25000|2024-01-02|
|          4| Sneha|Hyderabad|     108| Laptop|       1|55000|2024-01-06|
|          5|Vikram|    Delhi|     109|     TV|       1|80000|2024-01-03|
|          5|Vikram|    Delhi|     110| Mobile|       2|40000|2024-01-07|
+-----------+------+---------+--------

- Task 2: Create: revenue column

In [0]:
from pyspark.sql.functions import col
df_orders.withColumn("revenue", col("quantity") * col("price")).show()

+--------+-----------+-------+--------+-----+----------+-------+
|order_id|customer_id|product|quantity|price|order_date|revenue|
+--------+-----------+-------+--------+-----+----------+-------+
|     101|          1| Laptop|       2|50000|2024-01-01| 100000|
|     102|          1| Mobile|       1|20000|2024-01-03|  20000|
|     103|          2| Tablet|       3|30000|2024-01-02|  90000|
|     104|          2| Laptop|       1|45000|2024-01-05|  45000|
|     105|          3| Mobile|       2|15000|2024-01-01|  30000|
|     106|          3| Laptop|       1|60000|2024-01-04|  60000|
|     107|          4| Tablet|       2|25000|2024-01-02|  50000|
|     108|          4| Laptop|       1|55000|2024-01-06|  55000|
|     109|          5|     TV|       1|80000|2024-01-03|  80000|
|     110|          5| Mobile|       2|40000|2024-01-07|  80000|
+--------+-----------+-------+--------+-----+----------+-------+



# Aggregations (Day 5)
- Task 3: Total revenue per customer


In [0]:
from pyspark.sql.functions import sum,desc

df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").groupBy("customer_id").agg(sum("revenue").alias("total_revenue")).show()



+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|       120000|
|          2|       135000|
|          3|        90000|
|          4|       105000|
|          5|       160000|
+-----------+-------------+



- Task 4: Total revenue per city


In [0]:
from pyspark.sql.functions import sum, desc

df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").groupBy("city").agg(sum("revenue").alias("total_revenue")).show()


+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Kochi|       120000|
|  Chennai|       135000|
|Bangalore|        90000|
|Hyderabad|       105000|
|    Delhi|       160000|
+---------+-------------+



- Task 5 Total revenue per product

In [0]:
from pyspark.sql.functions import sum, desc

df_orders.withColumn("revenue", col("quantity") * col("price")).groupBy("product").agg(sum("revenue").alias("total_revenue")).show()

+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Laptop|       260000|
| Mobile|       130000|
| Tablet|       140000|
|     TV|        80000|
+-------+-------------+



# Ranking (Day 8)
- Task 6: Rank customers by revenue


In [0]:
from pyspark.sql.functions import sum, desc, rank
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "left").groupBy("customer_id").agg(sum("revenue").alias("total_revenue"))

df_ranked = df.withColumn("revenue_rank", rank().over(Window.orderBy(desc("total_revenue")))).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------+------------+
|customer_id|total_revenue|revenue_rank|
+-----------+-------------+------------+
|          5|       160000|           1|
|          2|       135000|           2|
|          1|       120000|           3|
|          4|       105000|           4|
|          3|        90000|           5|
+-----------+-------------+------------+



- Task 7: Top 2 customers per city


In [0]:
from pyspark.sql.functions import sum, desc,dense_rank
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").groupBy("city", "customer_id", "name").agg(sum("revenue").alias("total_revenue"))

df.withColumn("customer_rank", dense_rank().over(Window.partitionBy("city").orderBy(desc("total_revenue")))).filter(col("customer_rank") <= 2).show()


+---------+-----------+------+-------------+-------------+
|     city|customer_id|  name|total_revenue|customer_rank|
+---------+-----------+------+-------------+-------------+
|Bangalore|          3| Rahul|        90000|            1|
|  Chennai|          2| Meera|       135000|            1|
|    Delhi|          5|Vikram|       160000|            1|
|Hyderabad|          4| Sneha|       105000|            1|
|    Kochi|          1| Arjun|       120000|            1|
+---------+-----------+------+-------------+-------------+



- Task 8: Top product per city

In [0]:
from pyspark.sql.functions import sum, desc, dense_rank
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").groupBy("city", "product").agg(sum("revenue").alias("total_revenue"))

df.withColumn("product_rank",dense_rank().over(Window.partitionBy("city").orderBy(desc("total_revenue")))).filter(col("product_rank") == 1).show()

+---------+-------+-------------+------------+
|     city|product|total_revenue|product_rank|
+---------+-------+-------------+------------+
|Bangalore| Laptop|        60000|           1|
|  Chennai| Tablet|        90000|           1|
|    Delhi| Mobile|        80000|           1|
|    Delhi|     TV|        80000|           1|
|Hyderabad| Laptop|        55000|           1|
|    Kochi| Laptop|       100000|           1|
+---------+-------+-------------+------------+



# lag/lead (Day 9)
- Task 9: Add: previous revenue


In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).withColumn("previous_revenue", lag("revenue").over(Window.partitionBy("customer_id").orderBy("order_date"))).show()


+--------+-----------+-------+--------+-----+----------+-------+----------------+
|order_id|customer_id|product|quantity|price|order_date|revenue|previous_revenue|
+--------+-----------+-------+--------+-----+----------+-------+----------------+
|     101|          1| Laptop|       2|50000|2024-01-01| 100000|            NULL|
|     102|          1| Mobile|       1|20000|2024-01-03|  20000|          100000|
|     103|          2| Tablet|       3|30000|2024-01-02|  90000|            NULL|
|     104|          2| Laptop|       1|45000|2024-01-05|  45000|           90000|
|     105|          3| Mobile|       2|15000|2024-01-01|  30000|            NULL|
|     106|          3| Laptop|       1|60000|2024-01-04|  60000|           30000|
|     107|          4| Tablet|       2|25000|2024-01-02|  50000|            NULL|
|     108|          4| Laptop|       1|55000|2024-01-06|  55000|           50000|
|     109|          5|     TV|       1|80000|2024-01-03|  80000|            NULL|
|     110|      

- Task 10: Calculate: revenue difference

In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).withColumn("previous_revenue", lag("revenue").over(Window.partitionBy("customer_id").orderBy("order_date"))).withColumn("revenue_difference", col("revenue") - col("previous_revenue")).show()

+--------+-----------+-------+--------+-----+----------+-------+----------------+------------------+
|order_id|customer_id|product|quantity|price|order_date|revenue|previous_revenue|revenue_difference|
+--------+-----------+-------+--------+-----+----------+-------+----------------+------------------+
|     101|          1| Laptop|       2|50000|2024-01-01| 100000|            NULL|              NULL|
|     102|          1| Mobile|       1|20000|2024-01-03|  20000|          100000|            -80000|
|     103|          2| Tablet|       3|30000|2024-01-02|  90000|            NULL|              NULL|
|     104|          2| Laptop|       1|45000|2024-01-05|  45000|           90000|            -45000|
|     105|          3| Mobile|       2|15000|2024-01-01|  30000|            NULL|              NULL|
|     106|          3| Laptop|       1|60000|2024-01-04|  60000|           30000|             30000|
|     107|          4| Tablet|       2|25000|2024-01-02|  50000|            NULL|          

# Cumulative (Day 9)
- Task 11: Running total revenue per city

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").withColumn("running_revenue", sum("revenue").over(Window.partitionBy("city").orderBy("order_date"))).show()


+-----------+--------+-------+--------+-----+----------+-------+------+---------+---------------+
|customer_id|order_id|product|quantity|price|order_date|revenue|  name|     city|running_revenue|
+-----------+--------+-------+--------+-----+----------+-------+------+---------+---------------+
|          3|     105| Mobile|       2|15000|2024-01-01|  30000| Rahul|Bangalore|          30000|
|          3|     106| Laptop|       1|60000|2024-01-04|  60000| Rahul|Bangalore|          90000|
|          2|     103| Tablet|       3|30000|2024-01-02|  90000| Meera|  Chennai|          90000|
|          2|     104| Laptop|       1|45000|2024-01-05|  45000| Meera|  Chennai|         135000|
|          5|     109|     TV|       1|80000|2024-01-03|  80000|Vikram|    Delhi|          80000|
|          5|     110| Mobile|       2|40000|2024-01-07|  80000|Vikram|    Delhi|         160000|
|          4|     107| Tablet|       2|25000|2024-01-02|  50000| Sneha|Hyderabad|          50000|
|          4|     10

# Business Thinking
- Q1 Which customer is most valuable?


In [0]:
from pyspark.sql.functions import sum, desc

df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").groupBy("customer_id", "name").agg(sum("revenue").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+-----------+------+-------------+
|customer_id|  name|total_revenue|
+-----------+------+-------------+
|          5|Vikram|       160000|
+-----------+------+-------------+
only showing top 1 row


- Q2 Which city performs best?


In [0]:
from pyspark.sql.functions import sum, desc

df_orders.withColumn("revenue", col("quantity") * col("price")).join(df_customers, "customer_id", "inner").groupBy("city").agg(sum("revenue").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)


+-----+-------------+
| city|total_revenue|
+-----+-------------+
|Delhi|       160000|
+-----+-------------+
only showing top 1 row


- Q3 Which product should be promoted?


In [0]:
from pyspark.sql.functions import sum, desc

df_orders.withColumn("revenue", col("quantity") * col("price")).groupBy("product").agg(sum("revenue").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Laptop|       260000|
+-------+-------------+
only showing top 1 row


- Q4 Is revenue increasing over time?

In [0]:
from pyspark.sql.functions import col, sum, lag
from pyspark.sql.window import Window

df = df_orders.withColumn("revenue", col("quantity") * col("price")).groupBy("order_date").agg(sum("revenue").alias("total_revenue")).orderBy("order_date").withColumn("revenue_difference",col("total_revenue") - lag("total_revenue",1).over(Window.orderBy("order_date")))
df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-------------+------------------+
|order_date|total_revenue|revenue_difference|
+----------+-------------+------------------+
|2024-01-01|       130000|              NULL|
|2024-01-02|       140000|             10000|
|2024-01-03|       100000|            -40000|
|2024-01-04|        60000|            -40000|
|2024-01-05|        45000|            -15000|
|2024-01-06|        55000|             10000|
|2024-01-07|        80000|             25000|
+----------+-------------+------------------+



In [0]:

increasing = df.filter(col("revenue_difference") < 0).count() == 0
print(f"Revenue is increasing over time: {increasing}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Revenue is increasing over time: False
